In [1]:
from PyPDF2 import PdfReader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.chat_models import ChatOpenAI
from langchain.chains.question_answering import load_qa_chain
import warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv
import os
import openai

In [2]:
pdf = 'content/resume.pdf'
pdf_reader = PdfReader(pdf)
print(pdf_reader)

In [3]:
text = ''
for page in pdf_reader.pages:
    text += page.extract_text()

print(text)

Andre Sealy
New York, NY |(347)-461-7821 |andretsealy@gmail.com |/ewww.kidquant.com |/gtbkidquant
EDUCATION
Stevens Institute of Technology New York, NY
Masters of Science in Financial Engineering; Major GPA: 3.81 Sept 2024 - Present
Relevant Coursework: Stochastic Calculus, Pricing & Hedging, Probability Theory, Machine Learning, Deep Learning
Hunter College New York, NY
Bachelor of Arts in Mathematics; Major GPA: 3.68 Jan 2020 - Present
Relevant Coursework: Numerical Analysis, Real Analysis, Mathematical Statistics, Linear Algebra
Pace University New York, NY
Bachelor of Business Administration in Finance; Major GPA: 4.0 December 2018
Bachelor of Arts in Economics; Major GPA: 4.0
Honors: Beta Gamma Sigma, Golden Key Society
EXPERIENCE
America On Tech New York, NY
Data Science Instructor Nov 2023 - Present
•Facilitate and coordinate weekly lectures, lab projects, coding examples, graded quizzes and homework assignments.
• Design interactive weekly learning modules for more than 50 stu

In [4]:
# split the long text into small chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=700,
                                               chunk_overlap=200,
                                               length_function=len)

chunks = text_splitter.split_text(text)
chunks

['Andre Sealy\nNew York, NY |(347)-461-7821 |andretsealy@gmail.com |/ewww.kidquant.com |/gtbkidquant\nEDUCATION\nStevens Institute of Technology New York, NY\nMasters of Science in Financial Engineering; Major GPA: 3.81 Sept 2024 - Present\nRelevant Coursework: Stochastic Calculus, Pricing & Hedging, Probability Theory, Machine Learning, Deep Learning\nHunter College New York, NY\nBachelor of Arts in Mathematics; Major GPA: 3.68 Jan 2020 - Present\nRelevant Coursework: Numerical Analysis, Real Analysis, Mathematical Statistics, Linear Algebra\nPace University New York, NY\nBachelor of Business Administration in Finance; Major GPA: 4.0 December 2018\nBachelor of Arts in Economics; Major GPA: 4.0',
 'Pace University New York, NY\nBachelor of Business Administration in Finance; Major GPA: 4.0 December 2018\nBachelor of Arts in Economics; Major GPA: 4.0\nHonors: Beta Gamma Sigma, Golden Key Society\nEXPERIENCE\nAmerica On Tech New York, NY\nData Science Instructor Nov 2023 - Present\n•Faci

In [5]:
chunks[0]

'Andre Sealy\nNew York, NY |(347)-461-7821 |andretsealy@gmail.com |/ewww.kidquant.com |/gtbkidquant\nEDUCATION\nStevens Institute of Technology New York, NY\nMasters of Science in Financial Engineering; Major GPA: 3.81 Sept 2024 - Present\nRelevant Coursework: Stochastic Calculus, Pricing & Hedging, Probability Theory, Machine Learning, Deep Learning\nHunter College New York, NY\nBachelor of Arts in Mathematics; Major GPA: 3.68 Jan 2020 - Present\nRelevant Coursework: Numerical Analysis, Real Analysis, Mathematical Statistics, Linear Algebra\nPace University New York, NY\nBachelor of Business Administration in Finance; Major GPA: 4.0 December 2018\nBachelor of Arts in Economics; Major GPA: 4.0'

In [6]:

load_dotenv()
openai.api_key = os.environ["OPENAI_API_KEY"]

def openai_function(openai_api_key, chunks, analyze):

    # Using OpenAI service for embedding
    embeddings = OpenAIEmbeddings(openai_api_key=openai_api_key)

    # Facebook AI Similarity Search library help us to convert text data to numerical vector
    vectorstores = FAISS.from_texts(chunks, embedding=embeddings)

    # compares the query and chunks, enabling the selection of the top 'K' most similiar chunks based on their similarity scores.
    docs = vectorstores.similarity_search(query=analyze, k=3)

    # creates an OpenAI object, using the ChatGPT 4 
    llm = ChatOpenAI(model='gpt-4o', api_key=openai_api_key)

    # question-answering (QA) pipeline, making use of the load_qa_chain function
    chain = load_qa_chain(llm=llm, chain_type='stuff')

    response = chain.run(input_documents=docs, question=analyze)
    return response



In [7]:
def resume_summary(query_with_chunks):
    query = f''' need to detailed summarization of below resume and finally conclude them

                """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                {query_with_chunks}
                """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                '''
    return query

summary = resume_summary(query_with_chunks=chunks)
summary_result = openai_function(openai_api_key=openai.api_key, chunks=chunks, analyze=summary)
print(summary_result)

Andre Sealy is based in New York, NY, and can be reached at andretsealy@gmail.com or through his website www.kidquant.com. He is currently pursuing a Master of Science in Financial Engineering at the Stevens Institute of Technology with a major GPA of 3.81, covering courses like Stochastic Calculus, Pricing & Hedging, Probability Theory, Machine Learning, and Deep Learning. He is also enrolled at Hunter College for a Bachelor of Arts in Mathematics with a major GPA of 3.68, with coursework in Numerical Analysis, Real Analysis, Mathematical Statistics, and Linear Algebra. Previously, he completed a Bachelor of Business Administration in Finance and a Bachelor of Arts in Economics at Pace University, both with a perfect major GPA of 4.0, and was recognized with honors such as Beta Gamma Sigma and Golden Key Society.

In terms of experience, Andre is a Data Science Instructor at America On Tech, where he designs and facilitates learning modules in Machine Learning, Statistics, and Data Vi

In [ ]:
def resume_strength(query_with_chunks):
    query = f'''need to detailed analysis and explain of the strength of below resume and finally conclude them
                """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                {query_with_chunks}
                """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                '''
    return query

strengths = resume_strength(query_with_chunks=chunks)
strengths_result = openai_function(openai_api_key=openai.api_key, chunks=chunks, analyze=strengths)
print(strengths_result)

In [ ]:
def resume_weakness(query_with_chunks):
    query = f'''need to detailed analysis and explain of the weakness of below resume and how to improve make a better resume.

                """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                {query_with_chunks}
                """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                '''
    return query

weakness = resume_weakness(query_with_chunks=summary_result)
result_weakness = openai_function(openai_api_key=openai.api_key, chunks=chunks, analyze=weakness)
print(result_weakness)

In [ ]:
def job_title_suggestion(query_with_chunks):

    query = f''' what are the job roles i apply to likedin based on below?
                  
                  """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                  {query_with_chunks}
                  """""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""""
                '''
    return query

suggestion = job_title_suggestion(query_with_chunks=summary_result)
result_suggestion = openai_function(openai_api_key=openai.api_key, chunks=chunks, analyze=suggestion)
print(result_suggestion)